In [23]:
import pandas as pd
import numpy as np

# features = pd.read_csv("C:\DECODELABS INTERN\ecommerce_customer_features.csv")
# targets = pd.read_csv("C:\DECODELABS INTERN\ecommerce_customer_targets.csv")

# df = pd.merge(features, targets, on='Customer_ID')

# df.to_csv('customer_churn.csv', index=False)


In [24]:
df = pd.read_csv("C:\DECODELABS INTERN\customer_churn.csv")
df.head()

,Customer_ID,account_age_months,avg_order_value,total_orders,days_since_last_purchase,discount_usage_rate,return_rate,customer_support_tickets,loyalty_member,browsing_frequency_per_week,cart_abandonment_rate,product_review_score_avg,engagement_score,satisfaction_score,price_sensitivity_index,churned
0,0520df14-712d-4c69-a0c5-95a2e7dfc1ff,46,164.96,12,17,0.243,0.1720,0,No,6.1,0.430,5.00,6.58,9.43,3.7,No
1,a4013b3f-0688-4096-a194-6074be8ffec8,3,39.09,4,5,0.591,0.0808,1,No,4.1,0.183,4.44,6.25,8.50,6.9,No
2,eb870f2c-ed3d-4a21-a8ac-273fae69ea4f,29,37.42,8,47,0.212,0.1424,0,No,1.2,0.426,3.87,3.32,8.40,4.3,No
3,a7433451-8ea9-428a-9d80-679c6963b39f,35,62.64,9,3,0.699,0.0128,0,No,3.8,0.730,4.75,6.42,9.71,7.5,No
4,43f81935-49e3-44d3-94d1-5c4715738988,39,113.03,1,7,0.382,0.0232,0,No,5.4,0.613,5.00,6.48,9.92,5.0,No


[Raw Data Ingestion] 
       │
       ▼
[Inject Missingness] ────► (Creates <5%, 5-20%, and >20% gaps to simulate raw noise)
       │
       ▼
[Decision Matrix]    ────► (Applies dropna, Global Median, and KNN Imputation)
       │
       ▼
[Winsorization]      ────► (Calculates IQR bounds and clips outliers via np.clip)
       │
       ▼
[Feature Engineering] ───► (Calculates 3 brand new predictive columns)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Customer_ID                  6000 non-null   object 
 1   account_age_months           6000 non-null   int64  
 2   avg_order_value              6000 non-null   float64
 3   total_orders                 6000 non-null   int64  
 4   days_since_last_purchase     6000 non-null   int64  
 5   discount_usage_rate          6000 non-null   float64
 6   return_rate                  6000 non-null   float64
 7   customer_support_tickets     6000 non-null   int64  
 8   loyalty_member               6000 non-null   object 
 9   browsing_frequency_per_week  6000 non-null   float64
 10  cart_abandonment_rate        6000 non-null   float64
 11  product_review_score_avg     6000 non-null   float64
 12  engagement_score             6000 non-null   float64
 13  satisfaction_score

In [26]:
print(f"\nLoaded Matrix Shape (Rows, Columns): {df.shape}")


Loaded Matrix Shape (Rows, Columns): (6000, 16)


In [27]:
# Create an explicit, independent copy of the original dataframe in RAM to protect the raw source data from accidental corruption
df_messy = df.copy()

# Set a hardcoded random seed so that any random row selections calculate exactly the same indexes every time you rerun this cell
np.random.seed(42)

# --- THRESHOLD TARGET 1: Inject ~3% missingness (Target for Row Deletion Route: < 5%) ---
# Use .sample to randomly select exactly 3% (fraction = 0.03) of the row indices from the dataframe without replacement
index_drop_route = df_messy.sample(frac=0.03, random_state=42).index

# Locate those specific row indexes inside the 'avg_order_value' column and replace their numerical values with an empty NaN object
df_messy.loc[index_drop_route, 'avg_order_value'] = np.nan


# --- THRESHOLD TARGET 2: Inject ~12% missingness (Target for Median Imputation Route: 5% - 20%) ---
# Use .sample to randomly isolate exactly 12% (fraction = 0.12) of the row indices from the dataframe table layout
index_median_route = df_messy.sample(frac=0.12, random_state=42).index
df_messy.loc[index_median_route, 'engagement_score'] = np.nan

# --- THRESHOLD TARGET 3: Inject ~25% missingness (Target for KNN Imputation Route: > 20%) ---
# Use .sample to capture exactly 25% (fraction = 0.25) of the data rows to simulate heavy database pipeline corruption
index_knn_route = df_messy.sample(frac=0.25, random_state=42).index

# Overwrite the continuous numerical data inside the 'discount_usage_rate' column with NaN markers at those selected positions
df_messy.loc[index_knn_route, 'discount_usage_rate'] = np.nan


# --- PIPELINE INGESTION VERIFICATION LOGS ---
# Calculating total summation of null cells per feature and divide by the total dataframe length to compute true missing percentages
missing_profile = (df_messy.isnull().sum() / len(df_messy)) * 100

# Print an organized text dashboard header to visually track the newly engineered baseline flaws
print("--- ARTIFICIAL ENTERPRISE MISSINGNESS PROFILE INJECTED ---")

# Execute a loop to iterate through every column and its mapped missing percentage to print out an enterprise diagnostic scan
for feature_name, null_percentage in missing_profile.items():
    print(f"Feature: {feature_name:<28} | Null Data Rate: {null_percentage:>5.2f}%")

--- ARTIFICIAL ENTERPRISE MISSINGNESS PROFILE INJECTED ---
Feature: Customer_ID                  | Null Data Rate:  0.00%
Feature: account_age_months           | Null Data Rate:  0.00%
Feature: avg_order_value              | Null Data Rate:  3.00%
Feature: total_orders                 | Null Data Rate:  0.00%
Feature: days_since_last_purchase     | Null Data Rate:  0.00%
Feature: discount_usage_rate          | Null Data Rate: 25.00%
Feature: return_rate                  | Null Data Rate:  0.00%
Feature: customer_support_tickets     | Null Data Rate:  0.00%
Feature: loyalty_member               | Null Data Rate:  0.00%
Feature: browsing_frequency_per_week  | Null Data Rate:  0.00%
Feature: cart_abandonment_rate        | Null Data Rate:  0.00%
Feature: product_review_score_avg     | Null Data Rate:  0.00%
Feature: engagement_score             | Null Data Rate: 12.00%
Feature: satisfaction_score           | Null Data Rate:  0.00%
Feature: price_sensitivity_index      | Null Data Rate:  0.

In [28]:
# Create copy of our messy dataframe to document this specific stage of our cleaning sequence
df_cleaned = df_messy.copy()

# Recalculate the current total number of records in the dataframe before applying the deletion rule
total_rows_before = len(df_cleaned)

# compute the missing data percentages for all columns in this state of the dataframe
current_missing_rates = (df_cleaned.isnull().sum() / total_rows_before) * 100

print("--- EXECUTING ROUTE 1: ROW DELETION THRESHOLD (< 5%) ---")

# inspect every single column and its matching missing value rate dynamically
for target_column, null_rate in current_missing_rates.items():

    if 0.0 < null_rate < 5.0:
        print(f"Triggered: Column '{target_column}' has a minor missing rate of {null_rate:.2f}%.")
        
        # Apply .dropna engine to remove rows column contains NaN object
        # subset=[target_column] targets only this feature, and inplace=True updates the object memory directly
        df_cleaned.dropna(subset=[target_column], inplace=True)
        
        # Print a diagnostic confirmation message indicating the successful completion of the deletion protocol
        print(f"Action: Successfully eliminated rows containing null indicators inside '{target_column}'.")

# Calculate the new total number of records remaining in the memory-allocated dataframe table after dropping rows
total_rows_after = len(df_cleaned)

# Compute the absolute differences in rows to evaluate exactly how much data volume was sacrificed
dropped_rows_count = total_rows_before - total_rows_after

# Output a final engineering summary tracking the structural evolution and integrity of your dataset matrix
print(f"\nSummary: Row volume dropped from {total_rows_before} down to {total_rows_after} (Sacrificed {dropped_rows_count} rows).")

--- EXECUTING ROUTE 1: ROW DELETION THRESHOLD (< 5%) ---
Triggered: Column 'avg_order_value' has a minor missing rate of 3.00%.
Action: Successfully eliminated rows containing null indicators inside 'avg_order_value'.

Summary: Row volume dropped from 6000 down to 5820 (Sacrificed 180 rows).


In [29]:
# Programmatically compute the updated missing data percentages across all columns after the previous deletion stage
current_missing_rates = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100

# Print a clean descriptive header in your notebook output log to clearly mark the entry into Route 2 execution
print("--- EXECUTING ROUTE 2: GLOBAL MEDIAN IMPUTATION THRESHOLD (5% - 20%) ---")

# Begin an automated loop to check every single feature column and its current missing data calculation dynamically
for target_column, null_rate in current_missing_rates.items():
    
    # Evaluate if the feature column's missingness metrics fall directly inside the strict 5% to 20% enterprise boundary
    if 5.0 <= null_rate <= 20.0:
        
        # Display an intentional trace log mapping out exactly which feature triggered this specific median routing block
        print(f"Triggered: Column '{target_column}' has a moderate missing rate of {null_rate:.2f}%.")
        
        # Calculate the mathematical median value of the valid, non-null numeric cells currently present inside this column
        median_value = df_cleaned[target_column].median()
        
        # Print a diagnostic log showing the exact computed baseline value that will be used to resolve the missing gaps
        print(f"  Calculated robust Global Median value: {median_value:.4f}")
        
        # Use the vectorized .fillna engine to permanently overwrite all explicit NaN objects inside this column with the median value
        # inplace=True forces the Pandas execution engine to commit the matrix updates directly inside your system RAM
        df_cleaned[target_column].fillna(median_value, inplace=True)
        
        # Print a diagnostic confirmation message indicating the successful execution of the median replacement protocol
        print(f"  Action: Successfully imputed all empty records inside '{target_column}' using the calculated median.")

# Recalculate the missingness metrics right after the loop to verify that the targeted column has been successfully cleaned
updated_missing_rates = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100

# Confirm that the targeted feature has officially been drops down to a zero percent missing baseline in the active dataframe
print(f"\nVerification: Current missing rate for 'engagement_score' is now: {updated_missing_rates['engagement_score']:.2f}%")

--- EXECUTING ROUTE 2: GLOBAL MEDIAN IMPUTATION THRESHOLD (5% - 20%) ---
Triggered: Column 'engagement_score' has a moderate missing rate of 9.28%.
  Calculated robust Global Median value: 5.1000
  Action: Successfully imputed all empty records inside 'engagement_score' using the calculated median.

Verification: Current missing rate for 'engagement_score' is now: 0.00%


C:\Users\agarw\AppData\Local\Temp\ipykernel_3840\3959640801.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[target_column].fillna(median_value, inplace=True)


In [30]:
# Import the specialized KNNImputer class from scikit-learn's imputation library module
from sklearn.impute import KNNImputer

# Programmatically calculate the updated missingness metrics across all features after the row deletion and median imputation stages
current_missing_rates = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100

# Print an organized descriptive header in your notebook output log to mark the entry into Route 3 execution
print("--- EXECUTING ROUTE 3: K-NEAREST NEIGHBORS IMPUTATION THRESHOLD (> 20%) ---")

# Begin an automated loop to check every single feature column and its current missing data percentage dynamically
for target_column, null_rate in current_missing_rates.items():
    
    # Check if the feature column's missing data rate strictly exceeds the heavy 20% enterprise boundary threshold
    if null_rate > 20.0:
        
        # Display an intentional trace log mapping out exactly which feature triggered this multi-dimensional routing block
        print(f"Triggered: Column '{target_column}' has a severe missing rate of {null_rate:.2f}%.")
        
        # Isolate and extract all column names that have continuous numeric data types, as KNN calculates mathematical spatial distances
        numeric_columns = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
        
        # Print a diagnostic log showing the feature space array that the KNN calculation engine will evaluate
        print(f"  Isolating {len(numeric_columns)} numerical features to map coordinate dimensions for Euclidean distance calculation.")
        
        # Instantiate the KNNImputer engine setting n_neighbors to 5 to evaluate the 5 closest data rows in spatial proximity
        knn_engine = KNNImputer(n_neighbors=5)
        
        # Apply the .fit_transform engine over the numeric column slice to calculate distances and fill missing blocks via array operations
        # The engine expects a clean numeric matrix array and outputs a transformed, fully populated NumPy array
        imputed_numeric_array = knn_engine.fit_transform(df_cleaned[numeric_columns])
        
        # Overwrite the numeric feature coordinates in the active dataframe by wrapping the transformed array back into a Pandas structure
        df_cleaned[numeric_columns] = pd.DataFrame(imputed_numeric_array, columns=numeric_columns, index=df_cleaned.index)
        
        # Print a diagnostic confirmation message indicating the successful completion of the spatial estimation protocol
        print(f"  Action: Successfully resolved all empty records inside '{target_column}' via 5-Nearest Neighbors multi-dimensional estimation.")

# Recalculate the overall missingness profile across the entire dataframe to execute a final quality check evaluation
final_missing_check = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100

# Output an enterprise diagnostic summary tracking the absolute eradication of all missing value spaces from the active data matrix
print("\n--- FINAL MISSING DATA CHECK POST-DECISION MATRIX ---")
print(f"Total missing values remaining in the entire dataset: {final_missing_check.sum()}")

--- EXECUTING ROUTE 3: K-NEAREST NEIGHBORS IMPUTATION THRESHOLD (> 20%) ---
Triggered: Column 'discount_usage_rate' has a severe missing rate of 22.68%.
  Isolating 13 numerical features to map coordinate dimensions for Euclidean distance calculation.
  Action: Successfully resolved all empty records inside 'discount_usage_rate' via 5-Nearest Neighbors multi-dimensional estimation.

--- FINAL MISSING DATA CHECK POST-DECISION MATRIX ---
Total missing values remaining in the entire dataset: 0.0


In [31]:
# to track changes specifically for the outlier neutralization stage makign a copy
df_stabilized = df_cleaned.copy()

# Define a specific list of continuous numerical columns known to contain heavy skewed distributions or extreme outlier spikes
outlier_target_columns = ['avg_order_value', 'days_since_last_purchase', 'total_orders']

print("--- EXECUTING NON-PARAMETRIC IQR WINSORIZATION (OUTLIER CAPPING) ---\n")

for col_name in outlier_target_columns:

    q1 = df_stabilized[col_name].quantile(0.25)
    q3 = df_stabilized[col_name].quantile(0.75)

    iqr = q3 - q1
   
    lower_bounds = q1 - (1.5 * iqr)
    upper_bounds = q3 + (1.5 * iqr)
    
    high_outlier_count = (df_stabilized[col_name] > upper_bounds).sum()
    low_outlier_count = (df_stabilized[col_name] < lower_bounds).sum()
    
    print(f"Feature '{col_name}': bounds bounds = [{lower_bounds:.2f}, {upper_bounds:.2f}] | Outliers detected: High={high_outlier_count}, Low={low_outlier_count}")
    
    # Use NumPy's highly efficient, hardware-level vectorized .clip engine to compress extreme values into the bounds boundaries
    # Anything lower than lower_bounds becomes lower_bounds, and anything higher than upper_bounds becomes upper_bounds
    df_stabilized[col_name] = np.clip(df_stabilized[col_name], lower_bounds, upper_bounds)
    
    # diagnostic confirmation log confirming that the extreme variances have been safely neutralized in system memory
    print(f"  Action: Successfully Winsorized '{col_name}'. Extreme outliers have been clipped to the statistical boundss.")

# Display a final validation log statement confirming that the row count remained completely unchanged throughout this process
print(f"\nVerification: Outlier capping complete. Row count successfully preserved at exactly: {len(df_stabilized)} rows.")

--- EXECUTING NON-PARAMETRIC IQR WINSORIZATION (OUTLIER CAPPING) ---

Feature 'avg_order_value': bounds bounds = [-39.96, 185.52] | Outliers detected: High=271, Low=0
  Action: Successfully Winsorized 'avg_order_value'. Extreme outliers have been clipped to the statistical boundss.
Feature 'days_since_last_purchase': bounds bounds = [-39.00, 89.00] | Outliers detected: High=285, Low=0
  Action: Successfully Winsorized 'days_since_last_purchase'. Extreme outliers have been clipped to the statistical boundss.
Feature 'total_orders': bounds bounds = [-17.00, 31.00] | Outliers detected: High=178, Low=0
  Action: Successfully Winsorized 'total_orders'. Extreme outliers have been clipped to the statistical boundss.

Verification: Outlier capping complete. Row count successfully preserved at exactly: 5820 rows.


In [32]:
# to preserve the data state before feature generation
df_features = df_stabilized.copy()

print("--- EXECUTING DOMAIN-DRIVEN FEATURE ENGINEERING ---")

df_features['order_value_density'] = df_features['avg_order_value'] / (df_features['total_orders'] + 1)
df_features['inactivity_ratio'] = df_features['days_since_last_purchase'] / (df_features['account_age_months'] + 1)
df_features['support_friction_index'] = df_features['customer_support_tickets'] * df_features['cart_abandonment_rate']

print("  Successfully generated new predictive columns: [order_value_density, inactivity_ratio, support_friction_index]")

df_features.to_csv('processed_module1_output.csv', index=False)

print(f"\nSUCCESS: Final Clean Data Matrix Shape: {df_features.shape}")

--- EXECUTING DOMAIN-DRIVEN FEATURE ENGINEERING ---
  Successfully generated new predictive columns: [order_value_density, inactivity_ratio, support_friction_index]

SUCCESS: Final Clean Data Matrix Shape: (5820, 19)


MODULE 2 STARTED

In [33]:
# Load the cleaned, imputed, and stabilized dataset generated from Module 1 into a new notebook memory dataframe object
df_mod2_base = pd.read_csv('processed_module1_output.csv')

# Drop the 'Customer_ID' hash identifier column using the vectorized .drop function because unique strings offer no mathematical value to patterns
# axis=1 targets the vertical column array directly, allowing the pipeline to discard unnecessary non-predictive attributes
df_mod2_base.drop('Customer_ID', axis=1, inplace=True)

# Print an organized text header in your notebook output log to announce the start of the categorical transformation sequence
print("--- EXECUTING GEOMETRIC NOMINAL TRANSFORMATIONS ---")

# Isolate and extract all column names that match the 'object' or text string data type category for engineering transparency
text_categorical_columns = df_mod2_base.select_dtypes(include=['object']).columns.tolist()

# Print out a trace diagnostic log outlining exactly which nominal features have been captured for geometric encoding
print(f"Captured text-based features for encoding: {text_categorical_columns}")

# Execute highly optimized One-Hot Encoding across the entire dataframe matrix using the native pd.get_dummies engine
# columns=text_categorical_columns restricts the transformation mapping strictly to our nominal categorical variables
# drop_first=True completely eliminates structural multi-collinearity by dropping the first encoded category (Dummy Variable Trap evasion)
# dtype=int forces the newly generated binary flags to populate explicitly as 0 and 1 integer values instead of True/False booleans
df_encoded = pd.get_dummies(df_mod2_base, columns=text_categorical_columns, drop_first=True, dtype=int)

# Print a tracking confirmation message indicating the successful execution of our vectorized structural transformation
print("Action: Successfully expanded text categories into independent geometric binary coordinate dimensions.")

# Output the complete layout list of columns present in our transformed dataframe matrix to verify the binary transformation
print("\n--- UPDATED DATA MATRIX SCHEMA COLUMNS ---")
print(df_encoded.columns.tolist())

# Display the final matrix layout dimensions to show how the categorical features split while dropping their base columns
print(f"\nVerification: Current encoded matrix shape (Rows, Columns): {df_encoded.shape}")

--- EXECUTING GEOMETRIC NOMINAL TRANSFORMATIONS ---
Captured text-based features for encoding: ['loyalty_member', 'churned']
Action: Successfully expanded text categories into independent geometric binary coordinate dimensions.

--- UPDATED DATA MATRIX SCHEMA COLUMNS ---
['account_age_months', 'avg_order_value', 'total_orders', 'days_since_last_purchase', 'discount_usage_rate', 'return_rate', 'customer_support_tickets', 'browsing_frequency_per_week', 'cart_abandonment_rate', 'product_review_score_avg', 'engagement_score', 'satisfaction_score', 'price_sensitivity_index', 'order_value_density', 'inactivity_ratio', 'support_friction_index', 'loyalty_member_Yes', 'churned_Yes']

Verification: Current encoded matrix shape (Rows, Columns): (5820, 18)


In [ ]:
# Set a strict threshold coefficient; any feature pairs with a correlation above 0.80 will trigger our decider protocol
correlation_threshold = 0.80

target_label = 'churned_Yes'
corr_matrix = df_encoded.corr()
abs_corr_matrix = corr_matrix.abs()

# Generate a boolean upper-triangle mask to isolate only the top half of the symmetric matrix, ignoring the diagonal line
upper_triangle_mask = np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool)

# Apply the upper-triangle mask to the absolute correlation dataframe to clear out mirror-image duplicated records
upper_tri_dataframe = abs_corr_matrix.where(upper_triangle_mask)

# Initialize an empty set structure to hold the unique names of columns selected for pruning to prevent duplicate operations
columns_marked_for_deletion = set()

# Print an organized text dashboard header to visually log the multicollinearity evaluation process in real time
print("--- RUNNING INTELLIGENT MULTICOLLINEARITY DECIDER ENGINE ---")

# Execute an iteration sequence across the vertical column headers of our isolated upper triangle correlation matrix
for col in upper_tri_dataframe.columns:
    
    # Execute a nested sequence across the horizontal index row headers of the upper triangle correlation matrix
    for row in upper_tri_dataframe.index:

        # Check if the specific intersecting absolute correlation score strictly crosses our 0.80 enterprise threshold fence
        if upper_tri_dataframe.loc[row, col] > correlation_threshold:
            
            # Print an explicit log detailing the discovery of a highly collinear redundant feature pairing
            print(f"\nCollinear Pair Detected: '{row}' & '{col}' | Combined Score = {abs_corr_matrix.loc[row, col]:.4f}")
            
            # Look up the absolute correlation strength of the row feature relative to the target label column
            row_target_strength = abs_corr_matrix.loc[row, target_label]
            
            # Look up the absolute correlation strength of the column feature relative to the target label column
            col_target_strength = abs_corr_matrix.loc[col, target_label]
            
            # Display a diagnostic log showing the independent predictive value of both features compared to the target
            print(f"  Target Relationship Strength: '{row}' = {row_target_strength:.4f} vs. '{col}' = {col_target_strength:.4f}")
            
            # If the row feature has a lower correlation with the target than the column feature, mark the row feature for deletion
            if row_target_strength < col_target_strength:
                print(f"  [DECISION] Flagging '{row}' for removal. It provides a weaker predictive signal for Churn.")
                columns_marked_for_deletion.add(row)
            
            # Otherwise, if the column feature has a lower or equal correlation with the target, mark the column feature for deletion
            else:
                print(f"  [DECISION] Flagging '{col}' for removal. It provides a weaker predictive signal for Churn.")
                columns_marked_for_deletion.add(col)

# Convert the unique set structure of marked columns into a clean python list array for execution
drop_list = list(columns_marked_for_deletion)

# Apply the vectorized .drop function to permanently strip the weak, highly correlated columns away from the dataframe
df_pruned = df_encoded.drop(columns=drop_list)

# Print a final execution confirmation log listing all the features successfully eradicated from the pipeline memory
print(f"\nEradication Summary: Permanently dropped {len(drop_list)} redundant columns: {drop_list}")

# Output the updated matrix layout dimensions to show how lean and optimized your feature store matrix has become
print(f"Final Optimized Matrix Shape (Rows, Columns): {df_pruned.shape}")

--- RUNNING INTELLIGENT MULTICOLLINEARITY DECIDER ENGINE ---

Collinear Pair Detected: 'days_since_last_purchase' & 'engagement_score' | Combined Score = 0.8340
  Target Relationship Strength: 'days_since_last_purchase' = 0.7601 vs. 'engagement_score' = 0.6918
  [DECISION] Flagging 'engagement_score' for removal. It provides a weaker predictive signal for Churn.

Collinear Pair Detected: 'product_review_score_avg' & 'satisfaction_score' | Combined Score = 0.8221
  Target Relationship Strength: 'product_review_score_avg' = 0.0977 vs. 'satisfaction_score' = 0.1328
  [DECISION] Flagging 'product_review_score_avg' for removal. It provides a weaker predictive signal for Churn.

Collinear Pair Detected: 'discount_usage_rate' & 'price_sensitivity_index' | Combined Score = 0.8370
  Target Relationship Strength: 'discount_usage_rate' = 0.0223 vs. 'price_sensitivity_index' = 0.0227
  [DECISION] Flagging 'discount_usage_rate' for removal. It provides a weaker predictive signal for Churn.

Colline

In [35]:
# Create an independent copy of our pruned dataframe to document and isolate the feature scaling stage
df_scaled = df_pruned.copy()

# Isolate the continuous numerical features that require variance stabilization, explicitly leaving out binary flags and the target column
columns_to_scale = [
    'account_age_months', 'avg_order_value', 'total_orders', 
    'days_since_last_purchase', 'return_rate', 'customer_support_tickets', 
    'browsing_frequency_per_week', 'cart_abandonment_rate', 'satisfaction_score', 
    'price_sensitivity_index', 'order_value_density', 'inactivity_ratio'
]

# Print an organized descriptive text header to mark the official execution of the vectorized standardization engine
print("--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---")

# Execute high-speed vectorized Z-score scaling across all targeted continuous columns simultaneously without a single procedural loop
# df_scaled[columns_to_scale].mean() calculates a vector of column means using hardware-level optimization loops
# df_scaled[columns_to_scale].std() calculates a vector of column standard deviations using hardware-level optimization loops
# Pandas automatically broadcasts the subtraction and division matrix math across the entire dataframe structure in a single execution step
df_scaled[columns_to_scale] = (df_scaled[columns_to_scale] - df_scaled[columns_to_scale].mean()) / df_scaled[columns_to_scale].std()

# Print a diagnostic confirmation trace showing that the numerical metrics have been scaled successfully
print("  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.")

# Save the fully optimized, purely numerical, and scaled dataset out to a CSV file to act as your finalized Module 2 production artifact
df_scaled.to_csv('processed_module2_output.csv', index=False)

# Output a descriptive summary tracing the means and standard deviations of the scaled columns to confirm perfect normalization alignment
print("\n--- STANDARDIZATION SAMPLE VERIFICATION ---")
print(f"Mean of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].mean():.4f} (Approaching 0)")
print(f"Std deviation of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].std():.4f} (Exactly 1)")

# Print the finalized clean schema dimensions of the feature store data matrix
print(f"\nSUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: {df_scaled.shape}")

--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---
  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.

--- STANDARDIZATION SAMPLE VERIFICATION ---
Mean of 'avg_order_value' post-scaling: -0.0000 (Approaching 0)
Std deviation of 'avg_order_value' post-scaling: 1.0000 (Exactly 1)

SUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: (5820, 14)


In [36]:
# Create an independent copy of our pruned dataframe to document and isolate the feature scaling stage
df_scaled = df_pruned.copy()

# Isolate the continuous numerical features that require variance stabilization, explicitly leaving out binary flags and the target column
columns_to_scale = [
    'account_age_months', 'avg_order_value', 'total_orders', 
    'days_since_last_purchase', 'return_rate', 'customer_support_tickets', 
    'browsing_frequency_per_week', 'cart_abandonment_rate', 'satisfaction_score', 
    'price_sensitivity_index', 'order_value_density', 'inactivity_ratio'
]

# Print an organized descriptive text header to mark the official execution of the vectorized standardization engine
print("--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---")

# Execute high-speed vectorized Z-score scaling across all targeted continuous columns simultaneously without a single procedural loop
# df_scaled[columns_to_scale].mean() calculates a vector of column means using hardware-level optimization loops
# df_scaled[columns_to_scale].std() calculates a vector of column standard deviations using hardware-level optimization loops
# Pandas automatically broadcasts the subtraction and division matrix math across the entire dataframe structure in a single execution step
df_scaled[columns_to_scale] = (df_scaled[columns_to_scale] - df_scaled[columns_to_scale].mean()) / df_scaled[columns_to_scale].std()

# Print a diagnostic confirmation trace showing that the numerical metrics have been scaled successfully
print("  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.")

# Save the fully optimized, purely numerical, and scaled dataset out to a CSV file to act as your finalized Module 2 production artifact
df_scaled.to_csv('processed_module2_output.csv', index=False)

# Output a descriptive summary tracing the means and standard deviations of the scaled columns to confirm perfect normalization alignment
print("\n--- STANDARDIZATION SAMPLE VERIFICATION ---")
print(f"Mean of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].mean():.4f} (Approaching 0)")
print(f"Std deviation of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].std():.4f} (Exactly 1)")

# Print the finalized clean schema dimensions of the feature store data matrix
print(f"\nSUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: {df_scaled.shape}")

--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---
  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.

--- STANDARDIZATION SAMPLE VERIFICATION ---
Mean of 'avg_order_value' post-scaling: -0.0000 (Approaching 0)
Std deviation of 'avg_order_value' post-scaling: 1.0000 (Exactly 1)

SUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: (5820, 14)


In [37]:
# Create an independent copy of our pruned dataframe to document and isolate the feature scaling stage
df_scaled = df_pruned.copy()

# Isolate the continuous numerical features that require variance stabilization, explicitly leaving out binary flags and the target column
columns_to_scale = [
    'account_age_months', 'avg_order_value', 'total_orders', 
    'days_since_last_purchase', 'return_rate', 'customer_support_tickets', 
    'browsing_frequency_per_week', 'cart_abandonment_rate', 'satisfaction_score', 
    'price_sensitivity_index', 'order_value_density', 'inactivity_ratio'
]

# Print an organized descriptive text header to mark the official execution of the vectorized standardization engine
print("--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---")

# Execute high-speed vectorized Z-score scaling across all targeted continuous columns simultaneously without a single procedural loop
# df_scaled[columns_to_scale].mean() calculates a vector of column means using hardware-level optimization loops
# df_scaled[columns_to_scale].std() calculates a vector of column standard deviations using hardware-level optimization loops
# Pandas automatically broadcasts the subtraction and division matrix math across the entire dataframe structure in a single execution step
df_scaled[columns_to_scale] = (df_scaled[columns_to_scale] - df_scaled[columns_to_scale].mean()) / df_scaled[columns_to_scale].std()

# Print a diagnostic confirmation trace showing that the numerical metrics have been scaled successfully
print("  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.")

# Save the fully optimized, purely numerical, and scaled dataset out to a CSV file to act as your finalized Module 2 production artifact
df_scaled.to_csv('processed_module2_output.csv', index=False)

# Output a descriptive summary tracing the means and standard deviations of the scaled columns to confirm perfect normalization alignment
print("\n--- STANDARDIZATION SAMPLE VERIFICATION ---")
print(f"Mean of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].mean():.4f} (Approaching 0)")
print(f"Std deviation of 'avg_order_value' post-scaling: {df_scaled['avg_order_value'].std():.4f} (Exactly 1)")

# Print the finalized clean schema dimensions of the feature store data matrix
print(f"\nSUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: {df_scaled.shape}")

--- EXECUTING VECTORIZED Z-SCORE STANDARDIZATION ---
  Action: Successfully completed vectorized standardization. All continuous features now share Mean = 0 and Std = 1.

--- STANDARDIZATION SAMPLE VERIFICATION ---
Mean of 'avg_order_value' post-scaling: -0.0000 (Approaching 0)
Std deviation of 'avg_order_value' post-scaling: 1.0000 (Exactly 1)

SUCCESS: Module 2 Complete. Exported Clean Vector Matrix Shape: (5820, 14)


In [38]:
# Load the standardized, pruned numeric output from Module 2 into our notebook runtime environment
df_mod3_base = pd.read_csv('processed_module2_output.csv')

# Load the original raw dataset once again specifically to harvest the original 'Customer_ID' text column series
df_raw_source = pd.read_csv('customer_churn.csv')

# Pull the specific row indices from our clean Module 2 dataframe (which contains fewer rows due to the Route 1 deletion step)
valid_matrix_indices = df_mod3_base.index

# Align and extract the exact matching 'Customer_ID' column strings from the raw data using our valid index alignment template
matching_customer_ids = df_raw_source.loc[valid_matrix_indices, 'Customer_ID'].values

# Insert the harvested identifier strings directly into the front of our active dataframe as an explicit Feature Store Entity Key
df_mod3_base.insert(0, 'Customer_ID', matching_customer_ids)

# Print an organized text header to mark the initiation of our temporal event spine generation script
print("--- INJECTING TEMPORAL EVENT SPINE (TIMESTAMP GENERATION) ---")

# Establish a fixed starting datetime window baseline using the native Pandas timestamp generation engine
start_date_baseline = pd.Timestamp('2026-01-01 00:00:00')

# Generate an array of randomized hourly offsets bounded between 1 and 720 hours (simulating a 30-day historical window distribution)
# len(df_mod3_base) ensures we generate an individual unique random timestamp offset vector matching every active row precisely
np.random.seed(42)
random_hour_offsets = np.random.randint(1, 720, size=len(df_mod3_base))

# Use high-speed vectorized Pandas timedelta math to add the random hour offsets directly to our fixed calendar starting date baseline
generated_timestamps = start_date_baseline + pd.to_timedelta(random_hour_offsets, unit='H')

# Append the newly calculated date array back into our active dataframe schema under the official name 'event_timestamp'
df_mod3_base['event_timestamp'] = generated_timestamps

# Export this newly structured timeline matrix as a CSV file to act as the raw foundational asset for our upcoming feature registration
df_mod3_base.to_csv('processed_module3_spine_output.csv', index=False)

# Display a diagnostic summary layout showing the newly structured temporal columns alongside a sample customer row profile
print("\n--- TIMELINE MATRIX INTEGRITY CHECK ---")
print(df_mod3_base[['Customer_ID', 'event_timestamp', 'avg_order_value', 'churned_Yes']].head())

# Print out the finalized dimensional shape metrics to show that your entity key and timestamp have been bound successfully
print(f"\nSuccess: Spine Injected. Current Data Frame Shape (Rows, Columns): {df_mod3_base.shape}")

--- INJECTING TEMPORAL EVENT SPINE (TIMESTAMP GENERATION) ---

--- TIMELINE MATRIX INTEGRITY CHECK ---
                            Customer_ID     event_timestamp  avg_order_value  \
0  0520df14-712d-4c69-a0c5-95a2e7dfc1ff 2026-01-05 07:00:00         1.988312   
1  a4013b3f-0688-4096-a194-6074be8ffec8 2026-01-19 04:00:00        -0.880402   
2  eb870f2c-ed3d-4a21-a8ac-273fae69ea4f 2026-01-12 07:00:00        -0.918463   
3  a7433451-8ea9-428a-9d80-679c6963b39f 2026-01-05 11:00:00        -0.343672   
4  43f81935-49e3-44d3-94d1-5c4715738988 2026-01-04 00:00:00         0.804771   

   churned_Yes  
0            0  
1            0  
2            0  
3            0  
4            0  

Success: Spine Injected. Current Data Frame Shape (Rows, Columns): (5820, 16)


C:\Users\agarw\AppData\Local\Temp\ipykernel_3840\1009389446.py:28: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  generated_timestamps = start_date_baseline + pd.to_timedelta(random_hour_offsets, unit='H')


In [39]:
# Create an explicit dictionary object to act as our centralized, infrastructure-grade Feast Metadata Registry
feast_feature_store_registry = {}

# Define the central core business Entity Key tracking parameters inside our metadata store structure
feast_feature_store_registry['entity'] = {
    'name': 'Customer_ID',                     # Specify the primary unique tracking lookup hash field name
    'value_type': 'STRING',                    # Set the underlying datatype configuration requirement for the entity index
    'description': 'Unique enterprise identifier mapping a single consolidated customer profile'
}

# Define the global tracking timeline parameters inside our centralized metadata store configuration
feast_feature_store_registry['temporal_spine'] = {
    'timestamp_column': 'event_timestamp',     # Target the precise structural date column to track time vectors
    'value_type': 'DATETIME',                  # Document the field datatype configuration for chronological alignment
    'purpose': 'Provides the temporal baseline required to execute safe point-in-time joins without data leakage'
}

# --- DEFINE FEATURE VIEW 1: slow-changing user demographic properties ---
user_demographic_features = [
    'account_age_months',                      # Include total months the user has maintained an active account profile
    'loyalty_member_Yes'                       # Include the binary categorical tracking index representing program membership
]

# Register the demographic layout profile directly into our centralized Feast dictionary database
feast_feature_store_registry['feature_views'] = {
    'user_demographics_view': {
        'name': 'user_demographics',           # Establish the formal registration namespace identifier
        'features': user_demographic_features,  # Bind the explicit column tracking array defined right above
        'update_frequency': 'Daily Batch'      # Document infrastructure scheduling cadence parameters
    }
}

# --- DEFINE FEATURE VIEW 2: High-frequency user activity and transaction behaviors ---
user_behavioral_features = [
    'avg_order_value', 'total_orders', 'days_since_last_purchase', 
    'return_rate', 'customer_support_tickets', 'browsing_frequency_per_week', 
    'cart_abandonment_rate', 'satisfaction_score', 'price_sensitivity_index', 
    'order_value_density', 'inactivity_ratio'  # Bind all remaining standardized continuous and engineered metrics
]

# Append the behavioral layout view directly into our operational feature views metadata registry map
feast_feature_store_registry['feature_views']['user_behavioral_view'] = {
    'name': 'user_behavioral_activity',        # Assign the formal system architectural namespace identifier
    'features': user_behavioral_features,      # Link the continuous and engineered behavioral data columns
    'update_frequency': 'Near Real-Time Stream' # Document high-frequency telemetry system update capabilities
}

# Print an organized text summary dashboard to visually log your metadata registry setup
print("--- FEAST INFRASTRUCTURE METADATA REGISTRY COMPILED ---")
print(f"Registered Primary Entity Index: {feast_feature_store_registry['entity']['name']}")
print(f"Registered Timeline Spine Column: {feast_feature_store_registry['temporal_spine']['timestamp_column']}")

# Extract and iterate through our newly built views to confirm successful schema registration
for view_key, view_meta in feast_feature_store_registry['feature_views'].items():
    print(f"\n[VIEW REGISTERED]: {view_meta['name'].upper()}")
    print(f"  Operational Cadence: {view_meta['update_frequency']}")
    print(f"  Monitored Column Features Count: {len(view_meta['features'])}")
    print(f"  Columns List: {view_meta['features']}")

--- FEAST INFRASTRUCTURE METADATA REGISTRY COMPILED ---
Registered Primary Entity Index: Customer_ID
Registered Timeline Spine Column: event_timestamp

[VIEW REGISTERED]: USER_DEMOGRAPHICS
  Operational Cadence: Daily Batch
  Monitored Column Features Count: 2
  Columns List: ['account_age_months', 'loyalty_member_Yes']

[VIEW REGISTERED]: USER_BEHAVIORAL_ACTIVITY
  Operational Cadence: Near Real-Time Stream
  Monitored Column Features Count: 11
  Columns List: ['avg_order_value', 'total_orders', 'days_since_last_purchase', 'return_rate', 'customer_support_tickets', 'browsing_frequency_per_week', 'cart_abandonment_rate', 'satisfaction_score', 'price_sensitivity_index', 'order_value_density', 'inactivity_ratio']


In [40]:
# Create an explicit historical copy of our timeline dataframe from memory to build our simulated feature store core engine
df_feature_store = df_mod3_base.copy()

# Sort the feature store dataframe ascendingly by its event_timestamp, an absolute mandatory requirement for executing asof merge operations
df_store_sorted = df_feature_store.sort_values('event_timestamp')

# --- STEP 1: CONSTRUCT THE HISTORICAL OBSERVATION FRAME ---
# Isolate a random sample list of 5 unique customer ID strings from our database to simulate an incoming batch query request
np.random.seed(42)
sample_customer_query_list = df_store_sorted['Customer_ID'].sample(5, random_state=42).values

# Establish a strict point-in-time threshold cutoff date at the very end of January to capture their latest states up to that point
requested_query_timestamps = [pd.Timestamp('2026-02-01 00:00:00')] * 5

# Construct a fresh dataframe representing the incoming observation data framework query format from the modeling pipeline
observation_frame = pd.DataFrame({
    'Customer_ID': sample_customer_query_list,
    'requested_timestamp': requested_query_timestamps
})

# Sort the incoming observation query dataframe by its timestamp column to perfectly align the memory blocks for lookups
observation_frame_sorted = observation_frame.sort_values('requested_timestamp')

# Print an organized text dashboard header to visually log the point-in-time retrieval process in your notebook cell output
print("--- RUNNING POINT-IN-TIME HISTORICAL FEATURE RETRIEVAL QUERY ---")
print(f"Incoming Request: Fetching historical feature states for {len(observation_frame_sorted)} customers...")

# --- STEP 2: EXECUTE THE ZERO-LEAKAGE BACKWARD AS-OF MERGE ---
# Use the highly optimized pd.merge_asof vector machine to travel backwards through our time vectors instantly
# left_on maps our query timeline target, while right_on matches it against the feature store capture timeline
# by='Customer_ID' forces the engine to isolate and segment the historical timeline scan individually per customer profile
# direction='backward' guarantees zero data leakage by strictly pulling rows where event_timestamp <= requested_timestamp
retrieved_historical_features = pd.merge_asof(
    observation_frame_sorted,
    df_store_sorted,
    left_on='requested_timestamp',
    right_on='event_timestamp',
    by='Customer_ID',
    direction='backward'
)

# Print a tracking confirmation log signaling the successful compilation of our point-in-time correct training matrix
print("Action: Successfully completed retrospective timeline join. Future data cells have been strictly hidden.")

# Display the output result table to trace the requested timestamps against the exact historical event timestamps found
print("\n--- POINT-IN-TIME RETRIEVAL RESULTS DASHBOARD ---")
print(retrieved_historical_features[['Customer_ID', 'requested_timestamp', 'event_timestamp', 'avg_order_value', 'loyalty_member_Yes']])

# Export this point-in-time correct dataframe out to a CSV file to act as your finalized Module 3 enterprise feature asset
retrieved_historical_features.to_csv('final_production_feature_store_output.csv', index=False)

# Output a final confirmation message showing the clean layout dimensions of the retrieved training table asset
print(f"\nSUCCESS: Module 3 Complete. Final Feature Store Output Matrix Shape: {retrieved_historical_features.shape}")

--- RUNNING POINT-IN-TIME HISTORICAL FEATURE RETRIEVAL QUERY ---
Incoming Request: Fetching historical feature states for 5 customers...
Action: Successfully completed retrospective timeline join. Future data cells have been strictly hidden.

--- POINT-IN-TIME RETRIEVAL RESULTS DASHBOARD ---
                            Customer_ID requested_timestamp  \
0  f87b2141-5f28-4c7f-88c5-75751b158265          2026-02-01   
1  01cbaf0a-6424-4a7e-989c-5463ef620a6e          2026-02-01   
2  637a6395-74c0-4f99-91da-b617af5b4741          2026-02-01   
3  baf430a0-36a1-4347-9f54-97fc902aa5d8          2026-02-01   
4  c4ae508d-5204-4856-8911-910f7d22fb53          2026-02-01   

      event_timestamp  avg_order_value  loyalty_member_Yes  
0 2026-01-03 22:00:00        -0.915045                   1  
1 2026-01-10 16:00:00        -0.429594                   0  
2 2026-01-22 22:00:00        -1.439696                   0  
3 2026-01-28 22:00:00        -0.463553                   0  
4 2026-01-09 09:00:00  